# Load - capa golden
Se generan las tablas que consume el dashboard:
- `kpi_departamentos`: ventas, clientes y tasa de recompra por departamento
- `top_productos`: los 10 productos mas comprados de cada departamento
- `actividad_dia_hora`: ordenes por dia y hora
- `segmentos_clientes`: clasificacion de clientes segun su frecuencia de compra

In [0]:
%run ./00_utils

In [0]:
from pyspark.sql.window import Window

params = obtener_parametros()
CATALOGO = params["catalogo"]
SILVER, GOLDEN = f"{CATALOGO}.silver", f"{CATALOGO}.golden"

df_detalle = spark.table(f"{SILVER}.detalle_ordenes").cache()
df_ordenes = spark.table(f"{SILVER}.ordenes")
validar_no_vacio(df_detalle, "silver.detalle_ordenes")

## kpi_departamentos

In [0]:
try:
    # Ventana sin particion para obtener el total general
    ventana_total = Window.partitionBy()

    df_kpi = (df_detalle.groupBy("departamento")
        .agg(F.count("*").alias("total_productos_vendidos"),
             F.countDistinct("order_id").alias("total_ordenes"),
             F.countDistinct("user_id").alias("total_clientes"),
             F.countDistinct("product_id").alias("productos_distintos"),
             F.round(F.avg(F.col("es_recompra").cast("int")), 4).alias("tasa_recompra"))
        .withColumn("participacion_pct",
                    F.round(F.col("total_productos_vendidos") * 100 /
                            F.sum("total_productos_vendidos").over(ventana_total), 2))
        .withColumn("fecha_proceso", F.current_timestamp()))

    escribir_tabla(df_kpi, f"{GOLDEN}.kpi_departamentos")
except Exception as e:
    log(f"Error: {e}", "ERROR", "golden.kpi_departamentos")
    raise

## top_productos

In [0]:
try:
    ventana_depto = Window.partitionBy("departamento").orderBy(F.col("veces_comprado").desc(), F.col("nombre_producto"))

    df_top = (df_detalle.groupBy("departamento", "nombre_producto")
        .agg(F.count("*").alias("veces_comprado"),
             F.round(F.avg(F.col("es_recompra").cast("int")), 4).alias("tasa_recompra"))
        .withColumn("ranking", F.row_number().over(ventana_depto))
        .filter(F.col("ranking") <= 10)
        .withColumn("fecha_proceso", F.current_timestamp()))

    escribir_tabla(df_top, f"{GOLDEN}.top_productos")
except Exception as e:
    log(f"Error: {e}", "ERROR", "golden.top_productos")
    raise

## actividad_dia_hora

In [0]:
try:
    df_actividad = (df_detalle.groupBy("dia_semana_num", "dia_semana", "hora_dia", "franja_horaria")
        .agg(F.countDistinct("order_id").alias("total_ordenes"),
             F.count("*").alias("total_productos"))
        .withColumn("promedio_productos_orden",
                    F.round(F.col("total_productos") / F.col("total_ordenes"), 2))
        .withColumn("fecha_proceso", F.current_timestamp()))

    escribir_tabla(df_actividad, f"{GOLDEN}.actividad_dia_hora")
except Exception as e:
    log(f"Error: {e}", "ERROR", "golden.actividad_dia_hora")
    raise

## segmentos_clientes

In [0]:
try:
    # Cantidad de ordenes y dias promedio entre ordenes por cliente
    df_hist = (df_ordenes.groupBy("user_id")
        .agg(F.count("*").alias("total_ordenes"),
             F.round(F.avg(F.when(~F.col("es_primera_orden"), F.col("dias_desde_orden_previa"))), 2)
              .alias("promedio_dias_entre_ordenes")))

    df_compra = (df_detalle.groupBy("user_id")
        .agg(F.count("*").alias("total_productos"),
             F.countDistinct("order_id").alias("ordenes_con_detalle"),
             F.round(F.avg(F.col("es_recompra").cast("int")), 4).alias("tasa_recompra")))

    # Departamento en el que mas compra cada cliente
    ventana_cliente = Window.partitionBy("user_id").orderBy(F.col("cantidad").desc(), F.col("departamento"))
    df_favorito = (df_detalle.groupBy("user_id", "departamento").agg(F.count("*").alias("cantidad"))
        .withColumn("rn", F.row_number().over(ventana_cliente))
        .filter("rn = 1")
        .select("user_id", F.col("departamento").alias("departamento_favorito")))

    df_segmentos = (df_hist
        .join(df_compra, "user_id", "inner")
        .join(df_favorito, "user_id", "left")
        .withColumn("promedio_productos_orden",
                    F.round(F.col("total_productos") / F.col("ordenes_con_detalle"), 2))
        .withColumn("segmento",
                    F.when(F.col("total_ordenes") > 40, "Leal")
                     .when(F.col("total_ordenes") > 15, "Frecuente")
                     .when(F.col("total_ordenes") > 5, "Regular")
                     .otherwise("Ocasional"))
        .withColumn("fecha_proceso", F.current_timestamp()))

    escribir_tabla(df_segmentos, f"{GOLDEN}.segmentos_clientes")
except Exception as e:
    log(f"Error: {e}", "ERROR", "golden.segmentos_clientes")
    raise

In [0]:
df_detalle.unpersist()
display(spark.table(f"{GOLDEN}.kpi_departamentos").orderBy(F.col("total_productos_vendidos").desc()))